# 🛡️ Malware Classification with Deep Neural Networks
This notebook trains a multi-class deep neural network to classify malware into 8 categories using a polymorphic augmented dataset, then performs recursive feature elimination (RFE) for interpretability.

**Pipeline Overview:**
1. Load & explore the augmented training data
2. Build & train a deep neural network with dropout regularization
3. Visualize training accuracy and loss curves
4. Run RFE with Logistic Regression and Decision Tree to identify top 20 features

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping, ReduceLROnPlateau
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import RFE
from sklearn.preprocessing import LabelEncoder

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version : {tf.__version__}")
print(f"Keras version      : {keras.__version__}")

## 2. Load & Explore Data

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
TRAIN_PATH = 'dataset/augmented_train.csv'
VAL_PATH   = 'dataset/validation.csv'

# Load datasets
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)

print(f"Training set   : {train_df.shape[0]:,} rows × {train_df.shape[1]:,} cols")
print(f"Validation set : {val_df.shape[0]:,} rows × {val_df.shape[1]:,} cols")
train_df.head()

In [ ]:
# Basic statistics and class distribution
print("=== Class Distribution (Training) ===")
print(train_df.iloc[:, -1].value_counts().sort_index())
print("\n=== Missing Values ===")
print(train_df.isnull().sum().sum(), "total missing values")

## 3. Prepare Features & Labels

In [ ]:
NUM_FEATURES = 1858
NUM_CLASSES  = 8

# Training split
X_train = train_df.iloc[:, :-1].values.astype(np.float32)
y_train = to_categorical(train_df.iloc[:, -1].values, num_classes=NUM_CLASSES)

# Validation split
X_val = val_df.iloc[:, :-1].values.astype(np.float32)
y_val = to_categorical(val_df.iloc[:, -1].values, num_classes=NUM_CLASSES)

print(f"X_train : {X_train.shape}  |  y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}    |  y_val   : {y_val.shape}")

## 4. Build the Model

In [ ]:
def build_model(input_dim: int, num_classes: int) -> Sequential:
    """Build a 3-hidden-layer DNN with dropout regularization."""
    model = Sequential([
        # Hidden layer 1
        Dense(1000, activation='relu', kernel_initializer='glorot_uniform',
              input_shape=(input_dim,), name='dense_1'),
        Dropout(0.2, name='dropout_1'),

        # Hidden layer 2
        Dense(750, activation='relu', kernel_initializer='glorot_uniform', name='dense_2'),
        Dropout(0.2, name='dropout_2'),

        # Hidden layer 3
        Dense(500, activation='relu', kernel_initializer='glorot_uniform', name='dense_3'),
        Dropout(0.2, name='dropout_3'),

        # Output layer — softmax for multi-class
        Dense(num_classes, activation='softmax', name='output'),
    ], name='malware_dnn')
    return model


model = build_model(NUM_FEATURES, NUM_CLASSES)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 5. Train the Model

In [ ]:
# ── Callbacks ──────────────────────────────────────────────────────────────
callbacks = [
    TensorBoard(log_dir='./event_graph', histogram_freq=1,
                write_graph=True, write_images=True),

    # Stop early if val_loss doesn't improve for 15 epochs
    EarlyStopping(monitor='val_loss', patience=15,
                  restore_best_weights=True, verbose=1),

    # Halve LR when val_loss stagnates for 8 epochs
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=8, min_lr=1e-6, verbose=1),
]

print('Training on 5,000 polymorphic samples, validating on untouched real data...')
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=150,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1,
)

## 6. Evaluate & Visualize Training

In [ ]:
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print(f"Validation Loss     : {val_loss:.4f}")
print(f"Validation Accuracy : {val_acc * 100:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History', fontsize=14, fontweight='bold')

# ── Accuracy ──
axes[0].plot(history.history['accuracy'],     label='Train Acc',  color='steelblue')
axes[0].plot(history.history['val_accuracy'], label='Val Acc',    color='darkorange', linestyle='--')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy over Epochs')
axes[0].legend()
axes[0].grid(alpha=0.3)

# ── Loss ──
axes[1].plot(history.history['loss'],     label='Train Loss', color='steelblue')
axes[1].plot(history.history['val_loss'], label='Val Loss',   color='darkorange', linestyle='--')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss over Epochs')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to training_history.png')

## 7. Save the Model

In [ ]:
# Modern recommended format (.keras); also save legacy .h5 for compatibility
model.save('malware_classifier.keras')
model.save('adam.h5')  # legacy format retained for downstream compatibility
print('Model saved as malware_classifier.keras and adam.h5')

## 8. Feature Importance via Recursive Feature Elimination (RFE)

In [ ]:
# Encode labels as integers for sklearn estimators
encoder = LabelEncoder()
_X = train_df.iloc[:, :-1]
_y = encoder.fit_transform(train_df.iloc[:, -1])
feature_names = list(train_df.columns[:-1])

estimators = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=SEED),
    'Decision Tree'      : DecisionTreeClassifier(random_state=SEED),
}

TOP_N = 20
rfe_results = {}

for name, estimator in estimators.items():
    print(f"Running RFE with {name}...")
    rfe = RFE(estimator=estimator, n_features_to_select=TOP_N, verbose=0)
    rfe.fit(_X, _y)
    rfe_results[name] = rfe
    print(f"  ✔ Done")

In [ ]:
# Display top-20 features per estimator side by side
summary = {}
for name, rfe in rfe_results.items():
    selected = [f for f, s in zip(feature_names, rfe.support_) if s]
    summary[name] = selected
    print(f"\n=== Top {TOP_N} Features — {name} ===")
    for i, feat in enumerate(selected, 1):
        print(f"  {i:2d}. {feat}")

# Features agreed upon by both estimators
consensus = set(summary['Logistic Regression']) & set(summary['Decision Tree'])
print(f"\n=== Consensus Features (both models agree) — {len(consensus)} features ===")
for feat in sorted(consensus):
    print(f"  • {feat}")

In [ ]:
# Save the RFE summary to a CSV for further analysis
import itertools

rfe_df = pd.DataFrame({
    name: pd.Series(feats) for name, feats in summary.items()
})
rfe_df.index = rfe_df.index + 1  # 1-based rank
rfe_df.index.name = 'Rank'
rfe_df.to_csv('rfe_top_features.csv')
print('RFE results saved to rfe_top_features.csv')
rfe_df